In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


# Parameters
DATA_PATH = "TravelInsurancePrediction.csv"
TEST_SIZE = 0.2
RANDOM_STATE = 42
EPOCHS = 100
BATCH_SIZE = 32


# Reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)


# Load data
def load_data(path):
    data = pd.read_csv(path)
    print("Data loaded successfully.\n")
    return data


# Basic EDA
def basic_eda(data):
    print(data.head())
    print("\nInfo:\n")
    print(data.info())
    print("\nDescribe:\n")
    print(data.describe())
    print("\nMissing values:\n")
    print(data.isnull().sum())


# Visualization
def plot_distributions(data):
    cols = ["Age", "AnnualIncome", "FamilyMembers"]

    for col in cols:
        plt.figure()
        plt.hist(data[col], bins=30, edgecolor="black")
        plt.title(f"{col} Distribution")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.show()

    counts = data["TravelInsurance"].value_counts()
    plt.bar(["No", "Yes"], counts.values)
    plt.title("Travel Insurance Purchase")
    plt.show()


# Preprocessing
def preprocess_data(data):
    data = data.copy()

    # Handle missing values
    data.fillna(0, inplace=True)

    # Encode categorical variables
    categorical_cols = [
        "Employment Type",
        "GraduateOrNot",
        "FrequentFlyer",
        "EverTravelledAbroad"
    ]

    for col in categorical_cols:
        data[col] = pd.factorize(data[col])[0]

    # Target
    data["Target"] = data["TravelInsurance"]

    num_cols = ["Age", "AnnualIncome", "FamilyMembers", "ChronicDiseases"]
    features = num_cols + categorical_cols

    X = data[features].values
    y_raw = data["Target"].values
    y = to_categorical(y_raw, 2)

    return X, y, y_raw, features, num_cols


# Split and scale
def split_and_scale(X, y, y_raw, num_cols, feature_names):
    X_train, X_val, y_train, y_val, y_train_raw, y_val_raw = train_test_split(
        X, y, y_raw,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_raw
    )

    # Standardization AFTER split (prevents data leakage)
    scaler = StandardScaler()

    # Only scale numerical columns
    num_indices = [feature_names.index(col) for col in num_cols]

    X_train[:, num_indices] = scaler.fit_transform(X_train[:, num_indices])
    X_val[:, num_indices] = scaler.transform(X_val[:, num_indices])

    return X_train, X_val, y_train, y_val, y_train_raw, y_val_raw


# Model
def build_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(16, activation='relu'),
        Dense(2, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# Training
def train_model(model, X_train, y_train, X_val, y_val):
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=1
    )

    return history


# Evaluation
def evaluate_model(model, X_val, y_val, y_true_raw):
    y_pred_prob = model.predict(X_val)
    y_pred = np.argmax(y_pred_prob, axis=1)

    print("\nAccuracy:", accuracy_score(y_true_raw, y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(y_true_raw, y_pred))
    print("\nClassification Report:\n", classification_report(y_true_raw, y_pred))

    # ROC AUC
    y_prob_positive = y_pred_prob[:, 1]
    roc_auc = roc_auc_score(y_true_raw, y_prob_positive)
    print("\nROC AUC Score:", roc_auc)

    # ROC curve
    fpr, tpr, _ = roc_curve(y_true_raw, y_prob_positive)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.show()

    ConfusionMatrixDisplay.from_predictions(y_true_raw, y_pred)
    plt.title("Confusion Matrix")
    plt.show()

    return y_pred_prob


# Training plots
def plot_training(history):
    plt.plot(history.history['accuracy'], label='Train')
    plt.plot(history.history['val_accuracy'], label='Validation')
    plt.title("Accuracy")
    plt.legend()
    plt.show()

    plt.plot(history.history['loss'], label='Train')
    plt.plot(history.history['val_loss'], label='Validation')
    plt.title("Loss")
    plt.legend()
    plt.show()


# Feature Importance (approximation)
def feature_importance(X_val, y_pred_prob, feature_names):
    """
    NOTE:
    This is NOT true feature importance.
    It is a rough approximation based on model outputs.
    Neural networks are not inherently interpretable.
    """

    importance = X_val.T @ y_pred_prob[:, 1] / len(y_pred_prob)

    plt.figure(figsize=(10, 5))
    plt.bar(feature_names, importance)
    plt.xticks(rotation=45)
    plt.title("Feature Importance (Approximation)")
    plt.show()


# Main 
def main():
    set_seed(RANDOM_STATE)

    data = load_data(DATA_PATH)

    basic_eda(data)
    plot_distributions(data)

    X, y, y_raw, feature_names, num_cols = preprocess_data(data)

    X_train, X_val, y_train, y_val, y_train_raw, y_val_raw = split_and_scale(
        X, y, y_raw, num_cols, feature_names
    )

    model = build_model(X_train.shape[1])

    history = train_model(model, X_train, y_train, X_val, y_val)

    y_pred_prob = evaluate_model(model, X_val, y_val, y_val_raw)

    plot_training(history)

    feature_importance(X_val, y_pred_prob, feature_names)


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'tensorflow'